In [1]:
from git import Repo
import tempfile
import os

# URL of the repo you want to ingest
REPO_URL = "https://github.com/RWTH-EBC/AixLib.git"

# Clone into a temp folder
tmp_dir = tempfile.mkdtemp()
Repo.clone_from(REPO_URL, tmp_dir)
print(f"Cloned into {tmp_dir}")

Cloned into /var/folders/8w/nhzjf7wn3f7bxb6vlmbqbwfc0000gn/T/tmp0amr_wqx


In [3]:
import glob
import os

# 読み込み対象の拡張子
EXTENSIONS = [".md", ".py", ".txt", ".rst"]

def load_repo_texts(root_dir):
    docs = []
    for ext in EXTENSIONS:
        pattern = os.path.join(root_dir, "**", f"*{ext}")
        for path in glob.glob(pattern, recursive=True):
            # ファイルでなければスキップ
            if not os.path.isfile(path):
                continue
            # サイズが大きすぎるものはスキップ
            if os.path.getsize(path) > 1e6:
                continue
            try:
                with open(path, encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                docs.append({
                    "path": os.path.relpath(path, root_dir),
                    "content": text
                })
            except Exception as e:
                # 万が一の読み込みエラーも無視
                print(f"Warning: failed to read {path}: {e}")
    return docs

# 使い方
documents = load_repo_texts(tmp_dir)
print(f"Loaded {len(documents)} files")

Loaded 1023 files


In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = []
for doc in documents:
    texts = splitter.split_text(doc["content"])
    for i, txt in enumerate(texts):
        chunks.append({
            "id": f"{doc['path']}-{i}",
            "text": txt
        })

print(f"Created {len(chunks)} text chunks")

Created 23751 text chunks


In [9]:
import openai
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

openai.api_key = os.getenv("OPENAI_API_KEY")
embedder = OpenAIEmbeddings()

# texts: list of strings
texts = [c["text"] for c in chunks]
metadatas = [{"source": c["id"]} for c in chunks]

# Create FAISS index
index = FAISS.from_texts(texts, embedder, metadatas=metadatas)

In [8]:
!uv add tiktoken

Resolved 173 packages in 402ms                                       
Prepared 1 package in 45ms                                               
Installed 1 package in 1ms                                  
 + tiktoken==0.9.0


In [15]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

# ↓ すでに保存してある faiss_index フォルダを読み込む
embedder = OpenAIEmbeddings()
index = FAISS.load_local(
    "faiss_index",
    embedder,
    allow_dangerous_deserialization=True  # <-- これを追加
)

In [16]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

# LLM
llm = ChatOpenAI(model_name="gpt-4", temperature=0)

# RetrievalQA
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",         # or "map_reduce", "refine", etc.
    retriever=index.as_retriever(),
    return_source_documents=True
)

# Ask a question
query = "How does the authentication flow work in this project?"
result = qa(query)

print("Answer:\n", result["result"])
print("\nSource Chunks:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")

/var/folders/8w/nhzjf7wn3f7bxb6vlmbqbwfc0000gn/T/ipykernel_12999/728249638.py:5: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model_name="gpt-4", temperature=0)
/var/folders/8w/nhzjf7wn3f7bxb6vlmbqbwfc0000gn/T/ipykernel_12999/728249638.py:17: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa(query)
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n'

Answer:
 The provided context does not contain any information about an authentication flow in a project.

Source Chunks:
- AixLib/Resources/ReferenceResults/Dymola/AixLib_Fluid_HeatExchangers_Validation_HeaterCooler_T_dynamic.txt-39
- AixLib/Resources/ReferenceResults/Dymola/AixLib_Fluid_Sensors_Examples_HeatMeter.txt-4
- AixLib/Resources/ReferenceResults/Dymola/AixLib_Fluid_FixedResistances_Examples_FixedResistancesExplicit.txt-0
- AixLib/Resources/ReferenceResults/Dymola/AixLib_Fluid_HeatPumps_Validation_ReciprocatingWaterToWater_ScalingFactor.txt-1


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [18]:
import os
import tempfile
import glob
import re
from git import Repo
import networkx as nx

# ───────── 前段はそのまま ─────────
# 1. リポジトリをクローン、Modelica ファイル読み込み、継承グラフ構築、FAISS インデックス構築 …（省略）
#    → index: FAISS インデックス, G: NetworkX DiGraph, class_defs: クラス定義辞書

# ───────── ここから Grok 3 用クライアント ─────────
# 依存: pip install openai
from openai import OpenAI

# 環境変数から API キーを取得
api_key = os.getenv("GROQ_API_KEY")
if api_key is None:
    raise RuntimeError("環境変数 GROQ_API_KEY に API キーをセットしてください")

# xAI Grok 3 API クライアントを初期化
client = OpenAI(
    api_key=api_key,
    base_url="https://api.x.ai/v1",      # xAI のエンドポイント
)

# ─────────── 0. 事前準備 ───────────
# これまでに行った…
# ・G: 継承関係の NetworkX DiGraph
# ・index: FAISS インデックス
# ・class_defs: クラス定義辞書
# ・GraphRAGRetriever クラス定義
#   （前の回答のコードをそのまま）

# ─────────── 1. Retriever のインスタンス化 ───────────
retriever = GraphRAGRetriever(
    vector_index=index,  # 先に作成した FAISS インデックス
    graph=G,             # 先に作成した NetworkX グラフ
    k_sim=3,             # 類似度検索の上位何件を種として拾うか
    hops=1               # グラフの何ホップまで隣接ノードを拾うか
)

# ─────────── 2. Grok3 クライアントの初期化 ───────────
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv("XAI_API_KEY"),
    base_url="https://api.x.ai/v1",
)

# ─────────── 3. 質問関数 ───────────
def answer_query(query: str) -> str:
    # 1) Retriever でコンテキスト抽出
    ctx = retriever.get_context(query)

    # 2) メッセージ構築
    messages = [
        {
            "role": "system",
            "content": (
                "あなたはModelicaプロジェクトの専門家です。\n"
                "以下に関連クラスの定義があります。参照して回答してください。\n\n"
                + ctx
            ),
        },
        {"role": "user", "content": query},
    ]

    # 3) Grok-3 へリクエスト
    completion = client.chat.completions.create(
        model="x-ai/grok-3-beta",
        messages=messages,
        temperature=0,
    )
    return completion.choices[0].message.content

# ─────────── 4. 実行例 ───────────
if __name__ == "__main__":
    q = "ヒートポンプの Calibration クラスはどのように継承構造になっていますか？"
    print(answer_query(q))

NameError: name 'GraphRAGRetriever' is not defined